# Lesson 2: Tool Calling

Use Gemini function calling (via LlamaIndex) to choose tools **and** fill in tool arguments.

## Setup

In [24]:
from helper import get_google_api_key, configure_settings

GOOGLE_API_KEY = get_google_api_key()
llm, embed_model = configure_settings()

2026-07-14 10:47:38,370 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"
2026-07-14 10:47:39,030 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 10:47:39,074 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-14 10:47:39,075 - ERROR - Task was destroyed but it is pending!
task: <Task pending name='Task-262' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/worawatlawanont/Projects/sut-ai-sw-dev/.venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-263' coro=<Kernel.shell_main() running at /Users/worawatlawanont/Projects/sut-ai-sw-dev/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_c

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-14 10:48:01,917 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:48:02,225 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:48:02,572 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:48:02,941 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:48:03,397 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 10:48:03,440 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

In [25]:
import nest_asyncio

nest_asyncio.apply()

## 1. Define a Simple Tool

In [26]:
from llama_index.core.tools import FunctionTool

def add(x: int, y: int) -> int:
    """Adds two integers together."""
    return x + y


def mystery(x: int, y: int) -> int:
    """Mystery function that operates on top of two numbers."""
    return (x + y) * (x + y)


add_tool = FunctionTool.from_defaults(fn=add)
mystery_tool = FunctionTool.from_defaults(fn=mystery)

In [29]:
from llama_index.llms.google_genai import GoogleGenAI
from helper import DEFAULT_LLM_MODEL

llm = GoogleGenAI(model=DEFAULT_LLM_MODEL, api_key=GOOGLE_API_KEY, temperature=0)
response = llm.predict_and_call(
    [add_tool, mystery_tool],
    "Tell me the output of the mystery function on 2 and 9",
    verbose=True,
)
print(str(response))

2026-07-14 10:51:14,511 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"
2026-07-14 10:51:16,998 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


=== Calling Function ===
Calling function: mystery with args: {"x": 2, "y": 9}
=== Function Output ===
121
121


## 2. Define an Auto-Retrieval Tool

### Load Data

```bash
curl -L "https://openreview.net/pdf?id=VtmBAGCN7o" -o metagpt.pdf
```

In [30]:
# !curl -L "https://openreview.net/pdf?id=VtmBAGCN7o" -o metagpt.pdf

from helper import load_pdf

documents = load_pdf("metagpt.pdf")


In [32]:
len(documents)

29

In [33]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

In [34]:
len(nodes)

34

In [37]:
print(nodes[33].get_content(metadata_mode="all"))

page_label: 29
file_name: metagpt.pdf
file_path: metagpt.pdf
file_type: application/pdf
file_size: 16911937
creation_date: 2026-07-13
last_modified_date: 2026-07-13

Preprint
Table 9: Additional results of pure MetaGPT w/o feedback on SoftwareDev. Averages (Avg.) of 70 tasks are calculated and 10 randomly selected tasks are
included. ‘#’ denotes ‘The number of’, while ‘ID’ is ‘Task ID’.
ID Code statistics Doc statistics Cost statistics Cost of revision Code executability
#code files #lines of code #lines per code file #doc files #lines of doc #lines per doc file #prompt tokens #completion tokens time costs money costs
0 5.00 196.00 39.20 3.00 210.00 70.00 24087.00 6157.00 582.04 $ 1.09 1. TypeError 4
1 6.00 191.00 31.83 3.00 230.00 76.67 32517.00 6238.00 566.30 $ 1.35 1. TypeError 4
2 3.00 198.00 66.00 3.00 235.00 78.33 21934.00 6316.00 553.11 $ 1.04 1. lack
@app.route(’/’)
3
3 5.00 164 32.80 3.00 202.00 67.33 22951.00 5312.00 481.34 $ 1.01 1. PNG file miss-
ing 2. Compile bug
fixes
2


In [38]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex(nodes)
query_engine = vector_index.as_query_engine(similarity_top_k=2)

In [40]:
from llama_index.core.vector_stores import MetadataFilters

query_engine = vector_index.as_query_engine(
    similarity_top_k=2,
    filters=MetadataFilters.from_dicts(
        [
            {"key": "page_label", "value": "2"},
        ]
    ),
)

response = query_engine.query(
    "What are some high-level results of MetaGPT?",
)

2026-07-14 10:54:58,083 - INFO - AFC is enabled with max remote calls: 10.
2026-07-14 10:55:09,760 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


In [41]:
print(str(response))

MetaGPT achieved a new state-of-the-art (SoTA) in code generation benchmarks with Pass@1 scores of 85.9% and 87.7%. In experimental evaluations, it reached a 100% task completion rate, demonstrating efficiency in token and time costs as well as robustness. Additionally, compared to other frameworks like ChatDev, AgentVerse, LangChain, and AutoGPT, MetaGPT excels in providing extensive functionality and handling higher levels of software complexity.


In [42]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '2', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2026-07-13', 'last_modified_date': '2026-07-13'}


### Define the Auto-Retrieval Tool

In [43]:
from typing import List

from llama_index.core.vector_stores import FilterCondition


def vector_query(
    query: str,
    page_numbers: List[str],
) -> str:
    """Perform a vector search over an index.

    query (str): the string query to be embedded.
    page_numbers (List[str]): Filter by set of pages. Leave BLANK if we want
        to perform a vector search over all pages. Otherwise, filter by the
        set of specified pages.
    """
    metadata_dicts = [
        {"key": "page_label", "value": p} for p in page_numbers
    ]

    query_engine = vector_index.as_query_engine(
        similarity_top_k=2,
        filters=MetadataFilters.from_dicts(
            metadata_dicts,
            condition=FilterCondition.OR,
        ),
    )
    response = query_engine.query(query)
    return response


vector_query_tool = FunctionTool.from_defaults(
    name="vector_tool",
    fn=vector_query,
)

In [44]:
llm = GoogleGenAI(model=DEFAULT_LLM_MODEL, api_key=GOOGLE_API_KEY, temperature=0)
response = llm.predict_and_call(
    [vector_query_tool],
    "What are the high-level results of MetaGPT as described on page 2?",
    verbose=True,
)
print(str(response))

2026-07-14 10:58:48,241 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"
2026-07-14 10:58:51,563 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-14 10:58:51,619 - INFO - AFC is enabled with max remote calls: 10.


=== Calling Function ===
Calling function: vector_tool with args: {"page_numbers": ["2"], "query": "high-level results of MetaGPT"}


2026-07-14 10:59:03,335 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


=== Function Output ===
MetaGPT achieved a new state-of-the-art (SoTA) in code generation benchmarks, with Pass@1 scores of 85.9% and 87.7%. In experimental evaluations, it reached a 100% task completion rate, demonstrating efficiency in token and time costs as well as robustness. Additionally, MetaGPT stands out from other frameworks—such as LangChain, AutoGPT, ChatDev, and AgentVerse—by offering extensive functionality and the ability to handle higher levels of software complexity.
MetaGPT achieved a new state-of-the-art (SoTA) in code generation benchmarks, with Pass@1 scores of 85.9% and 87.7%. In experimental evaluations, it reached a 100% task completion rate, demonstrating efficiency in token and time costs as well as robustness. Additionally, MetaGPT stands out from other frameworks—such as LangChain, AutoGPT, ChatDev, and AgentVerse—by offering extensive functionality and the ability to handle higher levels of software complexity.


In [45]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '2', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2026-07-13', 'last_modified_date': '2026-07-13'}


## Let's add some other tools!

In [46]:
from llama_index.core import SummaryIndex
from llama_index.core.tools import QueryEngineTool

summary_index = SummaryIndex(nodes)
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=False,
)
summary_tool = QueryEngineTool.from_defaults(
    name="summary_tool",
    query_engine=summary_query_engine,
    description="Useful if you want to get a summary of MetaGPT",
)


In [47]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool],
    "What are the MetaGPT comparisons with ChatDev described on page 8?",
    verbose=True,
)
print(str(response))

2026-07-14 11:00:37,291 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-14 11:00:37,351 - INFO - AFC is enabled with max remote calls: 10.


=== Calling Function ===
Calling function: vector_tool with args: {"query": "MetaGPT comparisons with ChatDev", "page_numbers": ["8"]}


2026-07-14 11:01:07,008 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


=== Function Output ===
MetaGPT outperforms ChatDev on the SoftwareDev dataset across nearly all metrics. Key comparisons include:

*   **Executability:** MetaGPT achieved a score of 3.75, while ChatDev scored 2.25.
*   **Efficiency and Speed:** MetaGPT has a shorter running time of 503 seconds compared to ChatDev's 762 seconds.
*   **Productivity and Token Usage:** Although MetaGPT uses more total tokens (24,613 or 31,255) than ChatDev (19,292), it is more efficient in code generation, requiring only 124.3 to 126.5 tokens per line of code, whereas ChatDev requires 248.9 tokens.
*   **Code Statistics:** MetaGPT produces more total code lines (194.6 to 251.4) and more code files (4.6 to 5.1) than ChatDev, which produced 77.5 total lines and 1.9 files.
*   **Human Revision Cost:** MetaGPT has a lower human revision cost (0.83 to 2.25) compared to ChatDev's 2.5.

These advantages are attributed to MetaGPT's use of SOPs—including streamlined workflow, structured communication, and role-pla

In [22]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '8', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2026-07-13', 'last_modified_date': '2026-07-13'}


In [48]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool],
    "What is a summary of the paper?",
    verbose=True,
)
print(str(response))

2026-07-14 11:01:43,518 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-14 11:01:43,552 - INFO - AFC is enabled with max remote calls: 10.


=== Calling Function ===
Calling function: summary_tool with args: {"input": "MetaGPT"}


2026-07-14 11:02:12,747 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


=== Function Output ===
MetaGPT is a meta-programming framework designed for multi-agent collaboration based on Large Language Models (LLMs). It incorporates efficient human workflows by encoding Standardized Operating Procedures (SOPs) into prompt sequences, which streamlines workflows and allows agents with domain expertise to verify intermediate results and reduce errors.

The framework utilizes an assembly line paradigm to assign diverse roles to various agents, breaking down complex tasks into smaller subtasks. In a simulated software company, these roles include:
*   **Product Manager:** Conducts business analysis and creates Product Requirements Documents (PRDs).
*   **Architect:** Translates requirements into system design components, such as interface definitions, data structures, and file lists.
*   **Project Manager:** Handles task distribution.
*   **Engineer:** Executes the designated classes and functions.
*   **QA Engineer:** Formulates test cases to ensure code quality.